In [1]:
import requests
from bs4 import BeautifulSoup

In [2]:
response = requests.get("https://www.lmsal.com/solarsoft/latest_events_archive.html")
soup = BeautifulSoup(response.content, 'html.parser')

In [3]:
tables = soup.find('table', border='5', cellpadding='5', cellspacing='3')

In [4]:
# import requests
# from bs4 import BeautifulSoup
# import csv

# # Function to extract data from a row
# def extract_event_data(row):
#     cells = row.find_all('td')
#     snapshot_time = cells[0].text.strip()
#     first_event = cells[1].text.strip()
#     last_event = cells[2].text.strip()
#     num_events = cells[3].text.strip()
#     largest_event = cells[4].text.strip()
#     b_count = cells[5].text.strip()
#     c_count = cells[6].text.strip()
#     m_count = cells[7].text.strip()
#     x_count = cells[8].text.strip()
#     href = cells[0].find('a')['href'] if cells[0].find('a') else ''  # Get href if <a> tag exists
#     return {
#         'Snapshot Time': snapshot_time,
#         'First Event': first_event,
#         'Last Event': last_event,
#         'Number of Events': num_events,
#         'Largest Event': largest_event,
#         '#B': b_count,
#         '#C': c_count,
#         '#M': m_count,
#         '#X': x_count,
#         'Href': href,
#     }

# # Main scraping function
# def scrape_lmsal_events(url):
#     response = requests.get(url)
#     print("Gotten Response")
#     soup = BeautifulSoup(response.content, 'html.parser')
#     print("Gotten Soup")


#     # Find the table containing the events
#     event_table = soup.find('table', border='5', cellpadding='5', cellspacing='3')
#     print('Gotten Event Table')

#     if not event_table:
#         print("Event table not found")
#         return

#     # Extract data from each row of the table
#     event_data = []
#     for row in event_table.find_all('tr')[1:]:  # Skip the header row
#         print('New Row')
#         event_data.append(extract_event_data(row))

#     return event_data

# # URL of the page to scrape
# url = 'https://www.lmsal.com/solarsoft/latest_events_archive.html'

# # Scrape the data
# events_data = scrape_lmsal_events(url)

# # Write the data to a CSV file
# if events_data:
#     keys = events_data[0].keys()
#     with open('lmsal_events.csv', 'w', newline='') as output_file:
#         dict_writer = csv.DictWriter(output_file, fieldnames=keys)
#         dict_writer.writeheader()
#         dict_writer.writerows(events_data)
#     print("Data has been scraped and saved to 'lmsal_events.csv'")
# else:
#     print("No data scraped")


In [9]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# Function to extract data from a row
def extract_event_data(row):
    cells = row.find_all('td')
    snapshot_time = cells[0].text.strip()
    first_event = cells[1].text.strip()
    last_event = cells[2].text.strip()
    num_events = cells[3].text.strip()
    largest_event = cells[4].text.strip()
    b_count = cells[5].text.strip()
    c_count = cells[6].text.strip()
    m_count = cells[7].text.strip()
    x_count = cells[8].text.strip()
    href = cells[0].find('a')['href'] if cells[0].find('a') else ''
    return {
        'Snapshot Time': snapshot_time,
        'First Event': first_event,
        'Last Event': last_event,
        'Number of Events': num_events,
        'Largest Event': largest_event,
        '#B': b_count,
        '#C': c_count,
        '#M': m_count,
        '#X': x_count,
        'Href': href,
    }

# Function to extract data from the linked pages
def extract_linked_data(link):
    response = requests.get(link)
    soup = BeautifulSoup(response.content, 'html.parser')
    
    event_table = soup.find('table', border='2', cellpadding='5', cellspacing='2')
    if not event_table:
        print("Event table not found")
        return
    
    # Extract data from each row of the table
    event_data = []
    for row in event_table.find_all('tr')[1:]:  # Skip the header row
        try:
            cells = row.find_all('td')
            event_number = cells[0].text.strip()
            event_name = cells[1].text.strip()
            start_time = cells[2].text.strip()
            stop_time = cells[3].text.strip()
            peak_time = cells[4].text.strip()
            goes_class = cells[5].text.strip()
            derived_position = cells[6].text.strip()
            event_data.append({
                'Event#': event_number,
                'EName': event_name,
                'Start': start_time,
                'Stop': stop_time,
                'Peak': peak_time,
                'GOES Class': goes_class,
                'Derived Position': derived_position,
            })
        except Exception as e:
            print(e)
            print(link)
    return event_data

# Main scraping function
def scrape_lmsal_events(url):
    response = requests.get(url)
    print("Gotten Response")
    soup = BeautifulSoup(response.content, 'html.parser')
    print("Parsed Data")

    # Find the table containing the events
    event_table = soup.find('table', border='5', cellpadding='5', cellspacing='3')
    if not event_table:
        print("Event table not found")
        return

    # Extract data from each row of the main table
    event_data = []
    for row in event_table.find_all('tr')[1:]:  # Skip the header row
        print("New Row")
        event_row_data = extract_event_data(row)
        
        # Extract data from the linked page
        link = event_row_data['Href']
        if link:
            linked_data = extract_linked_data('https://www.lmsal.com/solarsoft/' + link)
            event_row_data['Linked Data'] = linked_data
        else:
            event_row_data['Linked Data'] = None
        
        event_data.append(event_row_data)
    
    return event_data

# URL of the main page to scrape
url = 'https://www.lmsal.com/solarsoft/latest_events_archive.html'

# Scrape the data
events_data = scrape_lmsal_events(url)

# Create a DataFrame from the scraped data
df = pd.DataFrame(events_data)

# Write the data to a CSV file
df.to_csv('lmsal_events_with_linked_data.csv', index=False)
print("Data has been scraped and saved to 'lmsal_events_with_linked_data.csv'")


Gotten Response
Parsed Data
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New 

New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row


New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row


New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row


New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row


New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
Event table not found
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
New Row
Ne

Event table not found
New Row
Event table not found
New Row
Event table not found
New Row
Event table not found
New Row
Event table not found
New Row
Event table not found
New Row
Event table not found
New Row
Event table not found
New Row
Event table not found
New Row
Event table not found
New Row
Event table not found
New Row
Event table not found
New Row
Event table not found
New Row
Event table not found
New Row
Event table not found
New Row
Event table not found
New Row
Event table not found
New Row
Event table not found
New Row
Event table not found
New Row
Event table not found
New Row
Event table not found
New Row
Event table not found
New Row
Event table not found
New Row
Event table not found
New Row
Event table not found
New Row
Event table not found
New Row
Event table not found
New Row
Event table not found
New Row
Event table not found
New Row
Event table not found
New Row
Event table not found
New Row
Event table not found
New Row
Event table not found
New Row
Event tabl

In [19]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import csv
from datetime import datetime

class LmsalETL:
    def __init__(self, url, csv_filename):
        self.url = url
        self.csv_filename = csv_filename

    def extract_event_data(self, row):
        cells = row.find_all('td')
        snapshot_time = cells[0].text.strip()
        first_event = cells[1].text.strip()
        last_event = cells[2].text.strip()
        num_events = cells[3].text.strip()
        largest_event = cells[4].text.strip()
        b_count = cells[5].text.strip()
        c_count = cells[6].text.strip()
        m_count = cells[7].text.strip()
        x_count = cells[8].text.strip()
        href = cells[0].find('a')['href'] if cells[0].find('a') else ''
        return {
            'Snapshot Time': snapshot_time,
            'First Event': first_event,
            'Last Event': last_event,
            'Number of Events': num_events,
            'Largest Event': largest_event,
            '#B': b_count,
            '#C': c_count,
            '#M': m_count,
            '#X': x_count,
            'Href': href,
        }

    def extract_linked_data(self, link):
        response = requests.get(link)
        soup = BeautifulSoup(response.content, 'html.parser')
        
        event_table = soup.find('table', border='2', cellpadding='5', cellspacing='2')
        if not event_table:
            print("Event table not found")
            return []
        
        event_data = []
        for row in event_table.find_all('tr')[1:]:  # Skip the header row
            try:
                cells = row.find_all('td')
                event_number = cells[0].text.strip()
                event_name = cells[1].text.strip()
                start_time = cells[2].text.strip()
                stop_time = cells[3].text.strip()
                peak_time = cells[4].text.strip()
                goes_class = cells[5].text.strip()
                derived_position = cells[6].text.strip()
                event_data.append({
                    'Event#': event_number,
                    'EName': event_name,
                    'Start': start_time,
                    'Stop': stop_time,
                    'Peak': peak_time,
                    'GOES Class': goes_class,
                    'Derived Position': derived_position,
                })
            except Exception as e:
                print(e)
                print(link)
        return event_data

    def get_all_records(self):
        response = requests.get(self.url)
        soup = BeautifulSoup(response.content, 'html.parser')

        event_table = soup.find('table', border='5', cellpadding='5', cellspacing='3')
        if not event_table:
            print("Event table not found")
            return

        event_data = []
        for row in event_table.find_all('tr')[1:]:  # Skip the header row
            event_row_data = self.extract_event_data(row)
            
            # Extract data from the linked page
            link = event_row_data['Href']
            if link:
                linked_data = self.extract_linked_data('https://www.lmsal.com/solarsoft/' + link)
                event_row_data['Linked Data'] = linked_data
            else:
                event_row_data['Linked Data'] = []
            
            event_data.append(event_row_data)

        df = pd.DataFrame(event_data)
        df.to_csv(self.csv_filename, index=False)
        print(f"All records and linked data have been scraped and saved to '{self.csv_filename}'")

    def get_latest_delta(self):
        # Read the previously stored CSV to get the last date
        try:
            df_prev = pd.read_csv(self.csv_filename)
            last_date = df_prev['Snapshot Time'].max()
            last_date = datetime.strptime(last_date, '%d-%b-%Y %H:%M')
        except FileNotFoundError:
            print("CSV file not found. Creating a new file.")
            df_prev = pd.DataFrame()
            last_date = datetime.min

        # Scrape the website
        response = requests.get(self.url)
        soup = BeautifulSoup(response.content, 'html.parser')

        event_table = soup.find('table', border='5', cellpadding='5', cellspacing='3')
        if not event_table:
            print("Event table not found")
            return

        event_data = []
        for row in event_table.find_all('tr')[1:]:  # Skip the header row
            event_row_data = self.extract_event_data(row)
            event_date = datetime.strptime(event_row_data['Snapshot Time'], '%d-%b-%Y %H:%M')

            # Extract data from the linked page
            link = event_row_data['Href']
            if link:
                linked_data = self.extract_linked_data('https://www.lmsal.com/solarsoft/' + link)
                event_row_data['Linked Data'] = linked_data
            else:
                event_row_data['Linked Data'] = []

            # Check if the event date is after the last date in the previous data
            if event_date > last_date:
                event_data.append(event_row_data)

        if event_data:
            df_new = pd.DataFrame(event_data)
            # Append the new data to the existing CSV file
            df_combined = pd.concat([df_prev, df_new], ignore_index=True)
            df_combined.to_csv(self.csv_filename, index=False)
            print(f"Latest delta records and linked data have been scraped and appended to '{self.csv_filename}'")
        else:
            print("No new records found. CSV file remains unchanged.")


In [20]:
# df[df['Linked Data'].isna()]['Number of Events'].values

In [21]:

# URL of the main page to scrape
url = 'https://www.lmsal.com/solarsoft/latest_events_archive.html'

# CSV filename to store the data
csv_filename = 'lmsal_events_with_linked_data_new.csv'

# Create an instance of the ETL class
lmsal_etl = LmsalETL(url, csv_filename)

# Get all records and linked data
lmsal_etl.get_all_records()

# # Get the latest delta records and linked data
# lmsal_etl.get_latest_delta()

Event table not found
Event table not found
Event table not found
Event table not found
Event table not found
Event table not found
Event table not found
Event table not found
Event table not found
Event table not found
Event table not found
list index out of range
https://www.lmsal.com/solarsoft/ssw/last_events-2013/last_events_20130708_2323/index.html
Event table not found
Event table not found
Event table not found
Event table not found
Event table not found
Event table not found
Event table not found
list index out of range
https://www.lmsal.com/solarsoft/ssw/last_events-2005/last_events_20050303_1312/index.html
list index out of range
https://www.lmsal.com/solarsoft/ssw/last_events-2005/last_events_20050302_1903/index.html
Event table not found
Event table not found
Event table not found
Event table not found
Event table not found
Event table not found
Event table not found
Event table not found
Event table not found
Event table not found
Event table not found
Event table not foun

In [22]:
# Get the latest delta records and linked data
lmsal_etl.get_latest_delta()

Event table not found
Event table not found
Event table not found
Event table not found
Event table not found
Event table not found
Event table not found
Event table not found
Event table not found
Event table not found
Event table not found
list index out of range
https://www.lmsal.com/solarsoft/ssw/last_events-2013/last_events_20130708_2323/index.html
Event table not found
Event table not found
Event table not found
Event table not found
Event table not found
Event table not found
Event table not found
list index out of range
https://www.lmsal.com/solarsoft/ssw/last_events-2005/last_events_20050303_1312/index.html
list index out of range
https://www.lmsal.com/solarsoft/ssw/last_events-2005/last_events_20050302_1903/index.html
Event table not found
Event table not found
Event table not found
Event table not found
Event table not found
Event table not found
Event table not found
Event table not found
Event table not found
Event table not found
Event table not found
Event table not foun